In [ ]:

"""
CS372 Mini Project 1: Classification
WFH Burnout Dataset - Burnout Risk Prediction
==============================================
ขั้นตอน:
1. โหลดและสำรวจข้อมูล
2. เตรียมข้อมูล (Feature Engineering, Encoding)
3. Feature Importance (Tree-based + Permutation)
4. Feature Selection (2/3 Rule)
5. แบ่งข้อมูล Train/Validation/Test (80/20)
6. สร้างโมเดล: kNN, XGBoost (RandomForest), Neural Network
7. Grid Search Hyperparameter Tuning
8. ประเมินผลและเปรียบเทียบโมเดล
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             f1_score, precision_score, recall_score)
from xgboost import XGBClassifier
import os

# ============================================================
# 1. โหลดข้อมูล
# ============================================================
print("=" * 60)
print("1. LOADING DATASET")
print("=" * 60)

df = pd.read_csv('https://raw.githubusercontent.com/surabodee-pha/Cs371_mini_project/refs/heads/main/wfh_burnout_dataset.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumns:\n{df.dtypes}")
print(f"\nTarget Distribution:\n{df['burnout_risk'].value_counts()}")
print(f"\nMissing Values:\n{df.isnull().sum()}")
print(f"\nDescriptive Stats:\n{df.describe()}")

# ============================================================
# 2. เตรียมข้อมูล
# ============================================================
print("\n" + "=" * 60)
print("2. DATA PREPARATION")
print("=" * 60)

df_clean = df.copy()

# ลบ user_id ออก (ไม่มีประโยชน์ในการทำนาย)
df_clean = df_clean.drop(columns=['user_id'])
df_clean = df_clean.dropna().reset_index(drop=True)

# Encode day_type (categorical)
le_day = LabelEncoder()
df_clean['day_type'] = le_day.fit_transform(df_clean['day_type'])
print(f"day_type classes: {le_day.classes_}")

# Encode target
le_target = LabelEncoder()
df_clean['burnout_risk_encoded'] = le_target.fit_transform(df_clean['burnout_risk'])
print(f"burnout_risk classes: {le_target.classes_}")

X = df_clean.drop(columns=['burnout_risk', 'burnout_risk_encoded'])
y = df_clean['burnout_risk_encoded']

print(f"\nFeatures used ({X.shape[1]}): {list(X.columns)}")
print(f"Target: burnout_risk (0=High, 1=Low, 2=Medium)")

# ============================================================
# 3. Feature Importance (Tree-based + Permutation - 2/3 Rule)
# ============================================================
print("\n" + "=" * 60)
print("3. FEATURE IMPORTANCE (Tree-based + Permutation)")
print("=" * 60)

X_fi, X_fi_val, y_fi, y_fi_val = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# ใช้ RandomForest เป็น Tree-based model (แทน XGBoost)
rf_full = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)
rf_full.fit(X_fi, y_fi)
baseline_acc = accuracy_score(y_fi_val, rf_full.predict(X_fi_val))
print(f"Baseline Accuracy (All Features): {baseline_acc:.4f}")

# Tree-based Importance (GINI)
tree_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_full.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

print(f"\nTree-based Feature Importance (Top 10):\n{tree_importance.head(10)}")

# 2/3 Rule
total_features = X.shape[1]
k = int(np.ceil((2/3) * total_features))
selected_features_fi = tree_importance.head(k)['feature'].tolist()
print(f"\nUsing 2/3 Rule → Selecting top {k} features out of {total_features}")

# Retrain with selected features
rf_selected = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1)
rf_selected.fit(X_fi[selected_features_fi], y_fi)
selected_acc = accuracy_score(y_fi_val, rf_selected.predict(X_fi_val[selected_features_fi]))
print(f"Accuracy after 2/3 feature selection: {selected_acc:.4f}")

# Permutation Importance
perm = permutation_importance(
    rf_selected, X_fi_val[selected_features_fi], y_fi_val,
    n_repeats=30, random_state=42, scoring='accuracy', n_jobs=-1
)

importance_df = pd.DataFrame({
    'feature': selected_features_fi,
    'perm_importance_mean': perm.importances_mean,
    'perm_importance_std': perm.importances_std
}).sort_values('perm_importance_mean', ascending=False).reset_index(drop=True)
importance_df['rank'] = importance_df.index + 1

print("\n" + "=" * 40)
print("Permutation Importance Ranking")
print("=" * 40)
print(importance_df.to_string(index=False))

top3 = importance_df.head(3)['feature'].tolist()
print(f"\nTop 3 Features: {top3}")

# Plot Feature Importance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Tree-based
axes[0].barh(tree_importance['feature'][::-1], tree_importance['importance'][::-1], color='steelblue')
axes[0].set_xlabel('Gini Importance')
axes[0].set_title('Tree-based Feature Importance (RandomForest)')
axes[0].axvline(x=0, color='black', linewidth=0.5)

# Permutation
labels = importance_df['rank'].astype(str) + ' - ' + importance_df['feature']
axes[1].barh(labels[::-1], importance_df['perm_importance_mean'][::-1], color='coral')
axes[1].set_xlabel('Permutation Importance (Accuracy Drop Mean)')
axes[1].set_title('Permutation Importance Ranking (2/3 Rule)')

plt.tight_layout()
plt.savefig('feature_importance_plot.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: feature_importance_plot.png")

# ============================================================
# 4. Feature Selection - Final Selected Features
# ============================================================
print("\n" + "=" * 60)
print("4. FEATURE SELECTION")
print("=" * 60)

# ใช้ permutation importance เลือก features ที่ mean > 0
final_features = importance_df[importance_df['perm_importance_mean'] > 0]['feature'].tolist()
if len(final_features) < 5:
    final_features = importance_df.head(8)['feature'].tolist()

print(f"Final Selected Features ({len(final_features)}): {final_features}")

X_final = X[final_features]

# ============================================================
# 5. Train/Validation/Test Split (80/20 with CV inside train)
# ============================================================
print("\n" + "=" * 60)
print("5. TRAIN / TEST SPLIT (80/20)")
print("=" * 60)

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set:  {X_test.shape[0]} samples")
print(f"Train class distribution: {pd.Series(y_train).value_counts().to_dict()}")
print(f"Test class distribution:  {pd.Series(y_test).value_counts().to_dict()}")

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ============================================================
# 6. โมเดล 1: kNN + Grid Search
# ============================================================
print("\n" + "=" * 60)
print("6. MODEL 1: k-Nearest Neighbors (kNN)")
print("=" * 60)

knn_param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn_gs = GridSearchCV(
    KNeighborsClassifier(),
    knn_param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
knn_gs.fit(X_train_scaled, y_train)

print(f"Best Parameters (kNN): {knn_gs.best_params_}")
print(f"Best CV Accuracy (kNN): {knn_gs.best_score_:.4f}")

knn_best = knn_gs.best_estimator_
knn_cv_scores = cross_val_score(knn_best, X_train_scaled, y_train, cv=cv, scoring='accuracy')
knn_test_pred = knn_best.predict(X_test_scaled)
knn_test_acc = accuracy_score(y_test, knn_test_pred)

print(f"\nCV Accuracy: {knn_cv_scores.mean():.4f} ± {knn_cv_scores.std():.4f}")
print(f"Test Accuracy: {knn_test_acc:.4f}")
print(f"Difference (CV - Test): {knn_cv_scores.mean() - knn_test_acc:.4f}")
print(f"\nClassification Report (Test):\n{classification_report(y_test, knn_test_pred, target_names=le_target.classes_)}")

# ============================================================
# 7. โมเดล 2: XGBoost-like (GradientBoosting) + Grid Search
# ============================================================
print("\n" + "=" * 60)
print("7. MODEL 2: Gradient Boosting (XGBoost-like)")
print("=" * 60)

# ลดจำนวน combinations เพื่อความเร็ว
gb_param_grid_reduced = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0]
}

gb_gs = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='mlogloss'),
    gb_param_grid_reduced,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
gb_gs.fit(X_train_scaled, y_train)

print(f"Best Parameters (XGBoost): {gb_gs.best_params_}")
print(f"Best CV Accuracy (XGBoost): {gb_gs.best_score_:.4f}")

gb_best = gb_gs.best_estimator_
gb_cv_scores = cross_val_score(gb_best, X_train_scaled, y_train, cv=cv, scoring='accuracy')
gb_test_pred = gb_best.predict(X_test_scaled)
gb_test_acc = accuracy_score(y_test, gb_test_pred)

print(f"\nCV Accuracy: {gb_cv_scores.mean():.4f} ± {gb_cv_scores.std():.4f}")
print(f"Test Accuracy: {gb_test_acc:.4f}")
print(f"Difference (CV - Test): {gb_cv_scores.mean() - gb_test_acc:.4f}")
print(f"\nClassification Report (Test):\n{classification_report(y_test, gb_test_pred, target_names=le_target.classes_)}")

# ============================================================
# 8. โมเดล 3: Neural Network (MLP) + Grid Search
# ============================================================
print("\n" + "=" * 60)
print("8. MODEL 3: Neural Network (MLP)")
print("=" * 60)

nn_param_grid = {
    'hidden_layer_sizes': [(64, 32), (128, 64), (64, 32, 16)],
    'activation': ['relu', 'tanh'],
    'learning_rate_init': [0.001, 0.01],
    'max_iter': [300]
}

nn_gs = GridSearchCV(
    MLPClassifier(random_state=42, early_stopping=True, validation_fraction=0.1),
    nn_param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
nn_gs.fit(X_train_scaled, y_train)

print(f"Best Parameters (NN): {nn_gs.best_params_}")
print(f"Best CV Accuracy (NN): {nn_gs.best_score_:.4f}")

nn_best = nn_gs.best_estimator_
nn_cv_scores = cross_val_score(nn_best, X_train_scaled, y_train, cv=cv, scoring='accuracy')
nn_test_pred = nn_best.predict(X_test_scaled)
nn_test_acc = accuracy_score(y_test, nn_test_pred)

print(f"\nCV Accuracy: {nn_cv_scores.mean():.4f} ± {nn_cv_scores.std():.4f}")
print(f"Test Accuracy: {nn_test_acc:.4f}")
print(f"Difference (CV - Test): {nn_cv_scores.mean() - nn_test_acc:.4f}")
print(f"\nClassification Report (Test):\n{classification_report(y_test, nn_test_pred, target_names=le_target.classes_)}")

# ============================================================
# 9. เปรียบเทียบโมเดลและสรุป
# ============================================================
print("\n" + "=" * 60)
print("9. MODEL COMPARISON SUMMARY")
print("=" * 60)

results = {
    'Model': ['kNN', 'XGBoost', 'Neural Network (MLP)'],
    'Best Parameters': [
        str(knn_gs.best_params_),
        str(gb_gs.best_params_),
        str(nn_gs.best_params_)
    ],
    'CV Accuracy (Mean)': [
        knn_cv_scores.mean(),
        gb_cv_scores.mean(),
        nn_cv_scores.mean()
    ],
    'CV Accuracy (Std)': [
        knn_cv_scores.std(),
        gb_cv_scores.std(),
        nn_cv_scores.std()
    ],
    'Test Accuracy': [
        knn_test_acc,
        gb_test_acc,
        nn_test_acc
    ],
    'Difference (CV-Test)': [
        knn_cv_scores.mean() - knn_test_acc,
        gb_cv_scores.mean() - gb_test_acc,
        nn_cv_scores.mean() - nn_test_acc
    ]
}

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Plot Comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models_short = ['kNN', 'XGBoost', 'MLP']
cv_means = [knn_cv_scores.mean(), gb_cv_scores.mean(), nn_cv_scores.mean()]
cv_stds = [knn_cv_scores.std(), gb_cv_scores.std(), nn_cv_scores.std()]
test_accs = [knn_test_acc, gb_test_acc, nn_test_acc]

# Bar chart comparison
x = np.arange(len(models_short))
width = 0.35
axes[0].bar(x - width/2, cv_means, width, label='CV Accuracy', color='steelblue',
            yerr=cv_stds, capsize=5)
axes[0].bar(x + width/2, test_accs, width, label='Test Accuracy', color='coral')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('CV vs Test Accuracy')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models_short)
axes[0].legend()
axes[0].set_ylim(0.8, 1.0)

# Confusion Matrices
for idx, (pred, name, ax) in enumerate(zip(
    [knn_test_pred, gb_test_pred, nn_test_pred],
    models_short,
    axes[1:]
)):
    if idx < 2:
        cm = confusion_matrix(y_test, pred)
        disp = ConfusionMatrixDisplay(cm, display_labels=le_target.classes_)
        disp.plot(ax=ax, colorbar=False)
        ax.set_title(f'Confusion Matrix - {name}')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved: model_comparison.png")

# Confusion matrix for all 3 models separately
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for pred, name, ax in zip(
    [knn_test_pred, gb_test_pred, nn_test_pred],
    ['kNN', 'XGBoost', 'MLP Neural Network'],
    axes
):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=le_target.classes_)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}')
plt.suptitle('Confusion Matrices - All Models (Test Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: confusion_matrices.png")

# ============================================================
# 10. สรุปผล
# ============================================================
print("\n" + "=" * 60)
print("10. CONCLUSION")
print("=" * 60)

best_idx = np.argmax(test_accs)
best_model_name = models_short[best_idx]
print(f"\nBest Model on Test Set: {best_model_name}")
print(f"Test Accuracy: {test_accs[best_idx]:.4f}")

for model, cv_m, cv_s, test, diff in zip(
    models_short, cv_means, cv_stds, test_accs,
    [knn_cv_scores.mean()-knn_test_acc,
     gb_cv_scores.mean()-gb_test_acc,
     nn_cv_scores.mean()-nn_test_acc]
):
    status = "ปกติ"
    if diff > 0.05:
        status = "มีแนวโน้ม Overfitting"
    elif cv_m < 0.7:
        status = "มีแนวโน้ม Underfitting"
    print(f"  {model}: CV={cv_m:.4f}±{cv_s:.4f}, Test={test:.4f}, Diff={diff:.4f} → {status}")

print("\nTop 3 Features (by Permutation Importance):")
for i, row in importance_df.head(3).iterrows():
    print(f"  Rank {row['rank']}: {row['feature']} (importance={row['perm_importance_mean']:.4f})")

print("\nFinal Selected Features:", final_features)
print("\n✓ Pipeline Completed Successfully")

1. LOADING DATASET
Dataset shape: (2000, 14)

Columns:
user_id                int64
day_type              object
work_hours           float64
screen_time_hours    float64
meetings_count         int64
breaks_taken           int64
after_hours_work       int64
app_switches           int64
sleep_hours          float64
task_completion      float64
isolation_index        int64
fatigue_score        float64
burnout_score        float64
burnout_risk          object
dtype: object

Target Distribution:
burnout_risk
Low       1019
Medium     843
High       138
Name: count, dtype: int64

Missing Values:
user_id              0
day_type             0
work_hours           0
screen_time_hours    0
meetings_count       0
breaks_taken         0
after_hours_work     0
app_switches         0
sleep_hours          0
task_completion      0
isolation_index      0
fatigue_score        0
burnout_score        0
burnout_risk         0
dtype: int64

Descriptive Stats:
           user_id   work_hours  screen_time_ho